# Loss Functions and Their Applications

This notebook provides a comprehensive overview of loss functions used in deep learning, including their mathematical foundations, implementations, and practical applications.

## 1. Import Required Libraries
We'll use the following libraries throughout this notebook:
- TensorFlow and Keras for deep learning implementations
- PyTorch as an alternative deep learning framework
- NumPy for numerical computations
- Matplotlib for visualizations
- Scikit-learn for utility functions

In [ ]:
# Import essential libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, mean_squared_error
import seaborn as sns

# Set plot style
plt.style.use('ggplot')

## 2. Understanding Loss Functions

Loss functions (also called cost functions or objective functions) are a crucial component in deep learning models. They quantify how well a model performs by measuring the difference between the model's predictions and the actual target values.

### Why Loss Functions Matter
- They provide a numerical measure of model performance
- They guide the optimization process during training
- They allow for backpropagation to update model weights
- Different loss functions are designed for different types of problems

### Key Characteristics of Loss Functions:
1. Non-negative values (the better the model performs, the closer to zero)
2. Differentiable (to enable gradient-based optimization)
3. Represent the "cost" of making incorrect predictions

### Mathematical Definition of Loss Functions

A loss function $L(y, \hat{y})$ measures the difference between true values $y$ and predicted values $\hat{y}$. For a dataset with $n$ samples:

- **Per-sample loss**: Loss computed for a single data point
- **Total loss**: Average or sum of losses over all data points

$$L_{total} = \frac{1}{n} \sum_{i=1}^{n} L(y_i, \hat{y}_i)$$

The goal of model training is to find parameters $\theta$ that minimize this loss function:

$$\theta^* = \arg\min_{\theta} L_{total}$$

## 3. Common Loss Functions for Regression

Regression problems involve predicting continuous values. The most common loss functions for regression tasks are:

1. Mean Squared Error (MSE)
2. Mean Absolute Error (MAE)
3. Huber Loss (combination of MSE and MAE)
4. Log-cosh Loss

Let's implement and analyze each of these loss functions.

In [ ]:
# Create some sample data for regression loss visualization
true_values = np.linspace(-2, 2, 100)
predictions_perfect = true_values.copy()
predictions_off = true_values + np.random.normal(0, 0.5, size=true_values.shape)

# Function to compute loss values for a range of predictions
def compute_loss_values(true_val, loss_func, prediction_range):
    losses = []
    for pred in prediction_range:
        y_true = np.full_like(prediction_range, true_val)
        y_pred = np.full_like(prediction_range, pred)
        loss = loss_func(y_true, y_pred)
        losses.append(loss)
    return np.array(losses)

### 3.1 Mean Squared Error (MSE)

MSE is one of the most common loss functions for regression. It measures the average squared difference between the predictions and the actual values.

**Mathematical Formula**: 
$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Properties**:
- Heavily penalizes large errors due to the squared term
- More sensitive to outliers
- Differentiable everywhere
- Convex function, ensuring a global minimum

In [ ]:
# MSE implementation
def mse_numpy(y_true, y_pred):
    return np.mean(np.square(y_true - y_pred))

# Calculate MSE using numpy and frameworks
mse_np = mse_numpy(true_values, predictions_off)
mse_tf = tf.keras.losses.MeanSquaredError()(true_values, predictions_off).numpy()
mse_torch = torch.nn.MSELoss()(torch.Tensor(predictions_off), torch.Tensor(true_values)).numpy()

print(f"MSE using NumPy: {mse_np:.4f}")
print(f"MSE using TensorFlow: {mse_tf:.4f}")
print(f"MSE using PyTorch: {mse_torch:.4f}")

# Visualize the MSE loss function
prediction_range = np.linspace(-3, 3, 100)
true_val = 1.0
mse_loss = compute_loss_values(true_val, mse_numpy, prediction_range)

plt.figure(figsize=(10, 6))
plt.plot(prediction_range, mse_loss, 'b-', linewidth=2)
plt.axvline(x=true_val, color='r', linestyle='--', label=f'True Value = {true_val}')
plt.grid(True)
plt.title('Mean Squared Error Loss', fontsize=15)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

### 3.2 Mean Absolute Error (MAE)

MAE measures the average absolute difference between the predictions and the actual values.

**Mathematical Formula**: 
$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Properties**:
- Less sensitive to outliers than MSE
- Provides constant gradients (not proportional to error magnitude)
- Not differentiable at zero, which can cause issues for gradient-based optimization
- More robust against outliers

In [ ]:
# MAE implementation
def mae_numpy(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Calculate MAE using numpy and frameworks
mae_np = mae_numpy(true_values, predictions_off)
mae_tf = tf.keras.losses.MeanAbsoluteError()(true_values, predictions_off).numpy()
mae_torch = torch.nn.L1Loss()(torch.Tensor(predictions_off), torch.Tensor(true_values)).numpy()

print(f"MAE using NumPy: {mae_np:.4f}")
print(f"MAE using TensorFlow: {mae_tf:.4f}")
print(f"MAE using PyTorch: {mae_torch:.4f}")

# Visualize the MAE loss function
mae_loss = compute_loss_values(true_val, mae_numpy, prediction_range)

plt.figure(figsize=(10, 6))
plt.plot(prediction_range, mae_loss, 'g-', linewidth=2)
plt.axvline(x=true_val, color='r', linestyle='--', label=f'True Value = {true_val}')
plt.grid(True)
plt.title('Mean Absolute Error Loss', fontsize=15)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

### 3.3 Huber Loss

Huber loss combines the best properties of MSE and MAE. It behaves like MSE for small errors and like MAE for large errors, making it less sensitive to outliers while still providing smooth gradients.

**Mathematical Formula**: 
$$
L_\delta(y, \hat{y}) = 
\begin{cases}
\frac{1}{2}(y - \hat{y})^2 & \text{for } |y - \hat{y}| \leq \delta \\
\delta(|y - \hat{y}| - \frac{1}{2}\delta) & \text{otherwise}
\end{cases}
$$

Where $\delta$ is a hyperparameter that controls the transition point between quadratic and linear behavior.

In [ ]:
# Huber loss implementation
def huber_numpy(y_true, y_pred, delta=1.0):
    error = y_true - y_pred
    abs_error = np.abs(error)
    quadratic = np.minimum(abs_error, delta)
    linear = abs_error - quadratic
    return np.mean(0.5 * quadratic**2 + delta * linear)

# Calculate Huber loss using numpy and frameworks
huber_np = huber_numpy(true_values, predictions_off)
huber_tf = tf.keras.losses.Huber(delta=1.0)(true_values, predictions_off).numpy()

print(f"Huber Loss using NumPy: {huber_np:.4f}")
print(f"Huber Loss using TensorFlow: {huber_tf:.4f}")

# Visualize Huber loss with different delta values
deltas = [0.5, 1.0, 2.0]
plt.figure(figsize=(12, 6))

for delta in deltas:
    def huber_with_delta(y_true, y_pred):
        return huber_numpy(y_true, y_pred, delta=delta)
    
    huber_loss = compute_loss_values(true_val, huber_with_delta, prediction_range)
    plt.plot(prediction_range, huber_loss, linewidth=2, label=f'δ = {delta}')

plt.axvline(x=true_val, color='r', linestyle='--', label=f'True Value = {true_val}')
plt.grid(True)
plt.title('Huber Loss with Different Delta Values', fontsize=15)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

# Compare MSE, MAE, and Huber on the same plot
plt.figure(figsize=(12, 6))
plt.plot(prediction_range, mse_loss, 'b-', linewidth=2, label='MSE')
plt.plot(prediction_range, mae_loss, 'g-', linewidth=2, label='MAE')
plt.plot(prediction_range, compute_loss_values(true_val, lambda y_true, y_pred: huber_numpy(y_true, y_pred, delta=1.0), prediction_range), 
         'y-', linewidth=2, label='Huber (δ=1.0)')
plt.axvline(x=true_val, color='r', linestyle='--', label=f'True Value = {true_val}')
plt.grid(True)
plt.title('Comparison of Regression Loss Functions', fontsize=15)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

### 3.4 Log-cosh Loss

Log-cosh loss is another function that, like Huber loss, is less sensitive to outliers but has the advantage of being twice differentiable everywhere.

**Mathematical Formula**:
$$L(y, \hat{y}) = \sum_{i=1}^{n} \log(\cosh(y_i - \hat{y}_i))$$

Where $\cosh(x) = \frac{e^x + e^{-x}}{2}$ is the hyperbolic cosine function.

**Properties**:
- Similar to MSE for small errors but less affected by outliers
- Smoother than Huber loss with easy-to-compute derivatives
- Second derivative is always defined, making it suitable for second-order optimization methods

In [ ]:
# Log-cosh loss implementation
def logcosh_numpy(y_true, y_pred):
    error = y_true - y_pred
    return np.mean(np.log(np.cosh(error)))

# Calculate Log-cosh loss
logcosh_np = logcosh_numpy(true_values, predictions_off)
logcosh_tf = tf.keras.losses.LogCosh()(true_values, predictions_off).numpy()

print(f"Log-cosh Loss using NumPy: {logcosh_np:.4f}")
print(f"Log-cosh Loss using TensorFlow: {logcosh_tf:.4f}")

# Visualize Log-cosh loss
logcosh_loss = compute_loss_values(true_val, logcosh_numpy, prediction_range)

plt.figure(figsize=(12, 6))
plt.plot(prediction_range, logcosh_loss, 'c-', linewidth=2, label='Log-cosh')
plt.plot(prediction_range, mse_loss, 'b-', linewidth=2, alpha=0.6, label='MSE')
plt.plot(prediction_range, mae_loss, 'g-', linewidth=2, alpha=0.6, label='MAE')
plt.axvline(x=true_val, color='r', linestyle='--', label=f'True Value = {true_val}')
plt.grid(True)
plt.title('Log-cosh Loss vs Other Regression Losses', fontsize=15)
plt.xlabel('Predicted Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

## 4. Common Loss Functions for Classification

Classification problems involve predicting categorical outcomes. The most common loss functions for classification tasks are:

1. Binary Cross-Entropy (for binary classification)
2. Categorical Cross-Entropy (for multi-class classification)
3. Sparse Categorical Cross-Entropy (variant for integer-encoded labels)
4. Hinge Loss (used in SVMs and margin-based classifiers)
5. Focal Loss (for handling class imbalance)

Let's implement and analyze each of these loss functions.

### 4.1 Binary Cross-Entropy Loss (Log Loss)

Binary cross-entropy is used for binary classification problems where the output is a probability value between 0 and 1.

**Mathematical Formula**:
$$BCE(y, \hat{y}) = -\frac{1}{n} \sum_{i=1}^{n} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

**Properties**:
- Measures the performance of a classification model whose output is a probability value
- For each observation, it penalizes the model based on how far the prediction is from the true value
- Provides larger gradients for confident but wrong predictions, pushing the model to correct them

In [ ]:
# Create binary classification sample data
np.random.seed(42)
num_samples = 1000
binary_true = np.random.randint(0, 2, num_samples)
binary_pred_good = np.clip(binary_true + np.random.normal(0, 0.3, num_samples), 0, 1)
binary_pred_random = np.random.random(num_samples)

# Binary cross-entropy implementation
def binary_crossentropy(y_true, y_pred, epsilon=1e-15):
    # Clip predictions to avoid log(0) or log(1)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Calculate BCE using numpy and frameworks
bce_np_good = binary_crossentropy(binary_true, binary_pred_good)
bce_np_random = binary_crossentropy(binary_true, binary_pred_random)
bce_tf_good = tf.keras.losses.BinaryCrossentropy()(binary_true, binary_pred_good).numpy()
bce_tf_random = tf.keras.losses.BinaryCrossentropy()(binary_true, binary_pred_random).numpy()

print(f"BCE for good predictions using NumPy: {bce_np_good:.4f}")
print(f"BCE for random predictions using NumPy: {bce_np_random:.4f}")
print(f"BCE for good predictions using TensorFlow: {bce_tf_good:.4f}")
print(f"BCE for random predictions using TensorFlow: {bce_tf_random:.4f}")

# Visualize BCE for different prediction probabilities
probabilities = np.linspace(0.0001, 0.9999, 100)
bce_class0 = [-np.log(1-p) for p in probabilities]  # True class = 0
bce_class1 = [-np.log(p) for p in probabilities]    # True class = 1

plt.figure(figsize=(10, 6))
plt.plot(probabilities, bce_class1, 'b-', linewidth=2, label='True Class = 1')
plt.plot(probabilities, bce_class0, 'r-', linewidth=2, label='True Class = 0')
plt.grid(True)
plt.title('Binary Cross-Entropy Loss', fontsize=15)
plt.xlabel('Predicted Probability for Class 1', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

### 4.2 Categorical Cross-Entropy Loss

Categorical cross-entropy is used for multi-class classification problems where the output is a probability distribution over multiple classes. It requires one-hot encoded labels.

**Mathematical Formula**:
$$CCE(y, \hat{y}) = -\sum_{i=1}^{n} \sum_{j=1}^{m} y_{ij} \log(\hat{y}_{ij})$$

Where $m$ is the number of classes, $y_{ij}$ is 1 if observation $i$ belongs to class $j$ and 0 otherwise, and $\hat{y}_{ij}$ is the predicted probability for observation $i$ belonging to class $j$.

**Properties**:
- Generalizes binary cross-entropy to multiple classes
- Works with one-hot encoded target vectors
- Provides larger penalties for confident but wrong predictions

In [ ]:
# Create multi-class classification sample data
num_classes = 5
num_samples = 1000

# Create one-hot encoded true labels
true_classes = np.random.randint(0, num_classes, num_samples)
y_true_onehot = np.zeros((num_samples, num_classes))
for i, c in enumerate(true_classes):
    y_true_onehot[i, c] = 1

# Create predicted probabilities (good model)
y_pred_good = np.zeros_like(y_true_onehot)
for i, c in enumerate(true_classes):
    # Add high probability for the correct class
    y_pred_good[i, c] = 0.7 + 0.2 * np.random.random()
    # Distribute remaining probability across other classes
    remaining = 1.0 - y_pred_good[i, c]
    other_classes = [j for j in range(num_classes) if j != c]
    for j in other_classes:
        if j == other_classes[-1]:
            y_pred_good[i, j] = remaining
        else:
            p = remaining * np.random.random()
            y_pred_good[i, j] = p
            remaining -= p

# Normalize to ensure probabilities sum to 1
y_pred_good = y_pred_good / y_pred_good.sum(axis=1, keepdims=True)

# Random predictions (uniform distribution)
y_pred_random = np.random.random((num_samples, num_classes))
y_pred_random = y_pred_random / y_pred_random.sum(axis=1, keepdims=True)

# Categorical cross-entropy implementation
def categorical_crossentropy(y_true, y_pred, epsilon=1e-15):
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]

# Calculate CCE using numpy and frameworks
cce_np_good = categorical_crossentropy(y_true_onehot, y_pred_good)
cce_np_random = categorical_crossentropy(y_true_onehot, y_pred_random)
cce_tf_good = tf.keras.losses.CategoricalCrossentropy()(y_true_onehot, y_pred_good).numpy()
cce_tf_random = tf.keras.losses.CategoricalCrossentropy()(y_true_onehot, y_pred_random).numpy()

print(f"CCE for good predictions using NumPy: {cce_np_good:.4f}")
print(f"CCE for random predictions using NumPy: {cce_np_random:.4f}")
print(f"CCE for good predictions using TensorFlow: {cce_tf_good:.4f}")
print(f"CCE for random predictions using TensorFlow: {cce_tf_random:.4f}")

# Visualize CCE for a single observation with 3 classes
true_label = [1, 0, 0]  # One-hot for class 0
plt.figure(figsize=(12, 6))

# Create a 2D probability space for the first two classes
# (third class probability is determined by the first two)
p1_range = np.linspace(0.01, 0.99, 30)
p2_range = np.linspace(0.01, 0.99, 30)
P1, P2 = np.meshgrid(p1_range, p2_range)

# Calculate CCE loss for all combinations
CCE = np.zeros_like(P1)
for i in range(len(p1_range)):
    for j in range(len(p2_range)):
        p1 = P1[i, j]
        p2 = P2[i, j]
        if p1 + p2 > 0.99:  # Skip invalid probability combinations
            CCE[i, j] = np.nan
        else:
            p3 = 1 - p1 - p2
            pred = [p1, p2, p3]
            CCE[i, j] = categorical_crossentropy(np.array([true_label]), np.array([pred]))

# Plot the surface
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')
valid_mask = ~np.isnan(CCE)
surf = ax.plot_surface(
    P1[valid_mask], P2[valid_mask], CCE[valid_mask], 
    cmap='viridis', edgecolor='none', alpha=0.8
)

# Add color bar
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, label='Loss Value')

ax.set_xlabel('Probability of Class 1')
ax.set_ylabel('Probability of Class 2')
ax.set_zlabel('CCE Loss')
ax.set_title('Categorical Cross-Entropy Loss for a 3-class Problem\n(True Class = 1)')

plt.tight_layout()
plt.show()

### 4.3 Sparse Categorical Cross-Entropy Loss

Sparse categorical cross-entropy is similar to categorical cross-entropy, but it works with integer class labels rather than one-hot encoded vectors.

**Mathematical Formula**:
$$SCCE(y, \hat{y}) = -\frac{1}{n}\sum_{i=1}^{n} \log(\hat{y}_{i, y_i})$$

Where $\hat{y}_{i, y_i}$ is the predicted probability for the true class $y_i$ of observation $i$.

**Properties**:
- Works directly with integer-encoded class labels
- Computationally efficient as it avoids one-hot encoding
- Otherwise identical to categorical cross-entropy

In [ ]:
# We'll use our previously defined true_classes (integer labels) and y_pred_good

# Sparse categorical cross-entropy implementation
def sparse_categorical_crossentropy(y_true_indices, y_pred, epsilon=1e-15):
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    loss = 0
    for i, true_idx in enumerate(y_true_indices):
        loss -= np.log(y_pred[i, true_idx])
    return loss / len(y_true_indices)

# Calculate SCCE using numpy and frameworks
scce_np_good = sparse_categorical_crossentropy(true_classes, y_pred_good)
scce_np_random = sparse_categorical_crossentropy(true_classes, y_pred_random)
scce_tf_good = tf.keras.losses.SparseCategoricalCrossentropy()(true_classes, y_pred_good).numpy()
scce_tf_random = tf.keras.losses.SparseCategoricalCrossentropy()(true_classes, y_pred_random).numpy()

print(f"SCCE for good predictions using NumPy: {scce_np_good:.4f}")
print(f"SCCE for random predictions using NumPy: {scce_np_random:.4f}")
print(f"SCCE for good predictions using TensorFlow: {scce_tf_good:.4f}")
print(f"SCCE for random predictions using TensorFlow: {scce_tf_random:.4f}")

# Compare with the regular categorical cross-entropy
print("\nComparison with categorical cross-entropy:")
print(f"CCE for good predictions: {cce_np_good:.4f}")
print(f"SCCE for good predictions: {scce_np_good:.4f}")
print(f"CCE for random predictions: {cce_np_random:.4f}")
print(f"SCCE for random predictions: {scce_np_random:.4f}")

### 4.4 Hinge Loss

Hinge loss is commonly used for maximum-margin classification problems, such as in Support Vector Machines. For binary classification with class labels -1 and 1, the hinge loss is defined as:

**Mathematical Formula**:
$$L(y, \hat{y}) = \max(0, 1 - y \cdot \hat{y})$$

For multi-class classification:
$$L(y, \hat{y}) = \sum_{j \neq y_i} \max(0, \hat{y}_j - \hat{y}_{y_i} + \Delta)$$

Where $\Delta$ is a margin parameter (typically 1).

**Properties**:
- Focuses on classifying examples correctly with a margin
- Does not provide probabilities like cross-entropy
- Non-differentiable at the hinge point
- Sparse in penalties - only wrong predictions within the margin contribute to the loss

In [ ]:
# Create binary classification data with -1 and 1 labels
n_samples = 100
binary_true_hinge = np.random.choice([-1, 1], size=n_samples)
binary_pred_hinge = np.random.uniform(-2, 2, size=n_samples)

# Hinge loss implementation for binary classification
def binary_hinge_loss(y_true, y_pred):
    return np.mean(np.maximum(0, 1 - y_true * y_pred))

# Calculate binary hinge loss
hinge_np = binary_hinge_loss(binary_true_hinge, binary_pred_hinge)
print(f"Hinge Loss using NumPy: {hinge_np:.4f}")

# Visualize Hinge Loss
x = np.linspace(-3, 3, 100)
plt.figure(figsize=(10, 6))
plt.plot(x, np.maximum(0, 1 - x), 'b-', linewidth=2, label='y_true = 1')
plt.plot(x, np.maximum(0, 1 + x), 'r-', linewidth=2, label='y_true = -1')
plt.grid(True)
plt.xlim(-3, 3)
plt.ylim(-0.5, 3)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.title('Hinge Loss', fontsize=15)
plt.xlabel('Decision Function Output (y_pred)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

# Compare Hinge Loss with Binary Cross-Entropy
plt.figure(figsize=(12, 6))
# For y_true = 1, convert decision function to probability using sigmoid
prob_x = 1 / (1 + np.exp(-x))  # convert to probability space
bce_loss = -np.log(prob_x)      # BCE when true label is 1
plt.plot(x, np.maximum(0, 1 - x), 'b-', linewidth=2, label='Hinge Loss (y_true = 1)')
plt.plot(x, bce_loss, 'g--', linewidth=2, label='BCE Loss (y_true = 1)')
plt.grid(True)
plt.xlim(-3, 3)
plt.ylim(-0.5, 5)
plt.title('Hinge Loss vs Binary Cross-Entropy', fontsize=15)
plt.xlabel('Decision Function Output (y_pred)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

### 4.5 Focal Loss

Focal loss is designed to address class imbalance problems by down-weighting easy examples and focusing on hard examples. It's a modified version of cross-entropy that adds a modulating factor.

**Mathematical Formula**:
$$FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

Where:
- $p_t$ is the model's estimated probability for the true class
- $\alpha_t$ is a balancing parameter for class weight
- $\gamma$ is the focusing parameter (typically 2)

**Properties**:
- Reduces the loss contribution from easy examples
- Increases the importance of correcting misclassified examples
- Helps handle class imbalance without resampling techniques
- Widely used in object detection

In [ ]:
# Focal loss implementation
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25, epsilon=1e-15):
    """
    Implementation of Focal Loss for binary classification
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    # Binary case
    pt = np.where(y_true == 1, y_pred, 1 - y_pred)
    
    # Alpha factor
    alpha_factor = np.where(y_true == 1, alpha, 1 - alpha)
    
    # Focusing factor
    focusing_factor = np.power(1 - pt, gamma)
    
    # Compute binary cross-entropy
    bce = -np.log(pt)
    
    # Compute focal loss
    loss = alpha_factor * focusing_factor * bce
    
    return np.mean(loss)

# Generate sample data with class imbalance
np.random.seed(42)
num_samples = 1000
# Create imbalanced dataset (90% class 0, 10% class 1)
imbalanced_true = np.random.choice([0, 1], size=num_samples, p=[0.9, 0.1])
# Generate predictions: good for class 0, random for class 1
imbalanced_pred = np.random.random(num_samples)
for i in range(num_samples):
    if imbalanced_true[i] == 0:
        # Better predictions for majority class
        imbalanced_pred[i] = np.random.beta(1, 5)  # tends toward 0
    else:
        # Random predictions for minority class
        imbalanced_pred[i] = np.random.random()

# Calculate losses
bce_imbalanced = binary_crossentropy(imbalanced_true, imbalanced_pred)
focal_imbalanced = focal_loss(imbalanced_true, imbalanced_pred)

print(f"Binary Cross-Entropy on imbalanced data: {bce_imbalanced:.4f}")
print(f"Focal Loss on imbalanced data: {focal_imbalanced:.4f}")

# Visualize how focal loss changes with gamma parameter
plt.figure(figsize=(10, 6))
prob_range = np.linspace(0.01, 0.99, 100)
y_true = 1  # Assume true class is 1
bce_loss = -np.log(prob_range)

gammas = [0, 1, 2, 5]
for gamma in gammas:
    if gamma == 0:
        # Regular cross-entropy
        focal = bce_loss
        label = 'BCE (γ=0)'
    else:
        focal = (1 - prob_range)**gamma * bce_loss
        label = f'Focal Loss (γ={gamma})'
    plt.plot(prob_range, focal, linewidth=2, label=label)

plt.grid(True)
plt.title('Focal Loss with Different Gamma Values\n(y_true = 1)', fontsize=15)
plt.xlabel('Predicted Probability for True Class', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.show()

## 5. Loss Functions for Special Use Cases

Beyond the standard regression and classification losses, many specialized loss functions have been developed for specific tasks:

1. IoU (Intersection over Union) Loss for object detection
2. Perceptual Loss for image generation
3. GAN Losses (adversarial, generator, discriminator)
4. Contrastive Loss for similarity learning
5. Triplet Loss for metric learning
6. CTC Loss for sequence-to-sequence learning

Let's implement and explore a few of these specialized loss functions.

### 5.1 IoU (Intersection over Union) Loss

IoU loss is commonly used in object detection models to measure the overlap between predicted and ground truth bounding boxes.

**Mathematical Formula**:
$$IoU = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

$$IoU Loss = 1 - IoU$$

**Properties**:
- Scale-invariant: doesn't depend on the size of the boxes
- Shape-invariant: only considers the overlapping regions
- Provides a better measure of localization than coordinate-based losses

In [ ]:
# IoU loss implementation for bounding boxes
def iou(box1, box2):
    """Calculate IoU between two bounding boxes.
    Each box is [x1, y1, x2, y2] where (x1,y1) is the top-left corner
    and (x2,y2) is the bottom-right corner."""
    
    # Calculate coordinates of the intersection
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    # Calculate area of intersection
    width_inter = max(0, x2_inter - x1_inter)
    height_inter = max(0, y2_inter - y1_inter)
    area_inter = width_inter * height_inter
    
    # Calculate areas of both boxes
    width_box1 = box1[2] - box1[0]
    height_box1 = box1[3] - box1[1]
    area_box1 = width_box1 * height_box1
    
    width_box2 = box2[2] - box2[0]
    height_box2 = box2[3] - box2[1]
    area_box2 = width_box2 * height_box2
    
    # Calculate area of union
    area_union = area_box1 + area_box2 - area_inter
    
    # Calculate IoU
    iou_value = area_inter / area_union if area_union > 0 else 0
    
    return iou_value

def iou_loss(box1, box2):
    """Calculate IoU loss between two bounding boxes."""
    return 1 - iou(box1, box2)

# Example: Calculate IoU and IoU Loss for two bounding boxes
ground_truth_box = [100, 100, 300, 300]  # [x1, y1, x2, y2]
prediction_box1 = [120, 120, 320, 320]   # Good prediction
prediction_box2 = [200, 200, 400, 400]   # Bad prediction

iou_value1 = iou(ground_truth_box, prediction_box1)
iou_value2 = iou(ground_truth_box, prediction_box2)
iou_loss1 = iou_loss(ground_truth_box, prediction_box1)
iou_loss2 = iou_loss(ground_truth_box, prediction_box2)

print(f"IoU for good prediction: {iou_value1:.4f}, IoU Loss: {iou_loss1:.4f}")
print(f"IoU for bad prediction: {iou_value2:.4f}, IoU Loss: {iou_loss2:.4f}")

# Visualize the bounding boxes and their IoU
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# First comparison
ax1.add_patch(plt.Rectangle((ground_truth_box[0], ground_truth_box[1]), 
                          ground_truth_box[2]-ground_truth_box[0], 
                          ground_truth_box[3]-ground_truth_box[1], 
                          fill=False, edgecolor='blue', linewidth=2, label='Ground Truth'))
ax1.add_patch(plt.Rectangle((prediction_box1[0], prediction_box1[1]), 
                          prediction_box1[2]-prediction_box1[0], 
                          prediction_box1[3]-prediction_box1[1], 
                          fill=False, edgecolor='red', linewidth=2, label='Prediction'))
ax1.set_title(f'Good Prediction: IoU = {iou_value1:.4f}')
ax1.legend()
ax1.set_xlim(0, 500)
ax1.set_ylim(0, 500)
ax1.invert_yaxis()  # To match image coordinates (origin at top-left)

# Second comparison
ax2.add_patch(plt.Rectangle((ground_truth_box[0], ground_truth_box[1]), 
                          ground_truth_box[2]-ground_truth_box[0], 
                          ground_truth_box[3]-ground_truth_box[1], 
                          fill=False, edgecolor='blue', linewidth=2, label='Ground Truth'))
ax2.add_patch(plt.Rectangle((prediction_box2[0], prediction_box2[1]), 
                          prediction_box2[2]-prediction_box2[0], 
                          prediction_box2[3]-prediction_box2[1], 
                          fill=False, edgecolor='red', linewidth=2, label='Prediction'))
ax2.set_title(f'Bad Prediction: IoU = {iou_value2:.4f}')
ax2.legend()
ax2.set_xlim(0, 500)
ax2.set_ylim(0, 500)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

### 5.2 Perceptual Loss

Perceptual loss is used in image generation tasks like style transfer or super-resolution. Instead of comparing pixels directly, it compares high-level features extracted from a pre-trained network.

**Key Concept**:
Compare the feature representations of images rather than their pixel values, which better aligns with human perception.

**Properties**:
- Better captures semantic information than pixel-wise losses
- Produces more visually pleasing results in image generation tasks
- Uses pre-trained networks (often VGG) as feature extractors

In [ ]:
# A simple demonstration of perceptual loss concept
# In practice, this would be implemented using a pre-trained model like VGG
# But we'll create a simplified version to illustrate the concept

# Create a simple "feature extractor" function
def simple_feature_extractor(image, layer_indices=[0, 1, 2]):
    """A simplified feature extractor that applies pooling operations"""
    features = {}
    current = image.copy()
    
    # Apply simple operations to simulate feature extraction
    for i in range(max(layer_indices) + 1):
        # Simple pooling to reduce resolution (simulating a CNN layer)
        if i > 0:
            current = current[::2, ::2]
        
        if i in layer_indices:
            features[f"layer_{i}"] = current
            
    return features

def perceptual_loss(image1, image2, layer_indices=[0, 1, 2], weights=[1.0, 1.0, 1.0]):
    """Calculate perceptual loss between two images using multiple layers"""
    features1 = simple_feature_extractor(image1, layer_indices)
    features2 = simple_feature_extractor(image2, layer_indices)
    
    total_loss = 0
    for i, layer in enumerate(layer_indices):
        layer_name = f"layer_{layer}"
        feat1 = features1[layer_name]
        feat2 = features2[layer_name]
        
        # MSE between features
        layer_loss = np.mean(np.square(feat1 - feat2))
        total_loss += weights[i] * layer_loss
        
    return total_loss

# Create sample images to test perceptual loss
# Original image
original = np.zeros((8, 8))
original[2:6, 2:6] = 1  # A simple square in the middle

# Pixel-shifted version (same pattern, slightly moved)
shifted = np.zeros((8, 8))
shifted[3:7, 3:7] = 1  # Square shifted by 1 pixel

# Noisy version (same position, but with noise)
noisy = original.copy()
np.random.seed(42)
noisy += np.random.normal(0, 0.2, size=original.shape)
noisy = np.clip(noisy, 0, 1)

# Different pattern (different semantic content)
different = np.zeros((8, 8))
different[1:7, 1:3] = 1  # A vertical rectangle
different[1:3, 3:7] = 1  # A horizontal rectangle

# Calculate MSE (pixel-wise) and perceptual losses
mse_shifted = np.mean(np.square(original - shifted))
mse_noisy = np.mean(np.square(original - noisy))
mse_different = np.mean(np.square(original - different))

perceptual_shifted = perceptual_loss(original, shifted)
perceptual_noisy = perceptual_loss(original, noisy)
perceptual_different = perceptual_loss(original, different)

print("Pixel-wise MSE:")
print(f"Original vs Shifted: {mse_shifted:.4f}")
print(f"Original vs Noisy: {mse_noisy:.4f}")
print(f"Original vs Different pattern: {mse_different:.4f}")
print("\nPerceptual Loss:")
print(f"Original vs Shifted: {perceptual_shifted:.4f}")
print(f"Original vs Noisy: {perceptual_noisy:.4f}")
print(f"Original vs Different pattern: {perceptual_different:.4f}")

# Visualize the images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(shifted, cmap='gray')
axes[1].set_title('Shifted')
axes[2].imshow(noisy, cmap='gray')
axes[2].set_title('Noisy')
axes[3].imshow(different, cmap='gray')
axes[3].set_title('Different Pattern')

for ax in axes:
    ax.axis('off')
    
plt.suptitle('Comparison of Images for Perceptual Loss', fontsize=16)
plt.tight_layout()
plt.show()

## 6. Custom Loss Functions

One of the advantages of modern deep learning frameworks is the ability to define custom loss functions tailored to specific tasks. Here we'll demonstrate how to create custom loss functions in TensorFlow and PyTorch.

Some common reasons to create custom loss functions:
1. Combining multiple loss functions with different weights
2. Implementing novel loss functions from research papers
3. Adding specific constraints or regularization terms
4. Creating task-specific losses that better represent the problem

In [ ]:
# Custom loss function in TensorFlow
import tensorflow as tf

# Example 1: Weighted combination of MSE and MAE
class WeightedMSEMAE(tf.keras.losses.Loss):
    def __init__(self, mse_weight=0.5, mae_weight=0.5, name="weighted_mse_mae"):
        super().__init__(name=name)
        self.mse_weight = mse_weight
        self.mae_weight = mae_weight
        
    def call(self, y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        mae = tf.reduce_mean(tf.abs(y_true - y_pred))
        return self.mse_weight * mse + self.mae_weight * mae

# Example 2: Custom loss using lambda function
def weighted_mse_mae_lambda(mse_weight=0.5, mae_weight=0.5):
    def loss(y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        mae = tf.reduce_mean(tf.abs(y_true - y_pred))
        return mse_weight * mse + mae_weight * mae
    return loss

# Test the custom losses
y_true = tf.constant([1.0, 2.0, 3.0])
y_pred = tf.constant([1.2, 1.8, 2.9])

# Using the class-based loss
weighted_loss = WeightedMSEMAE(mse_weight=0.7, mae_weight=0.3)
loss_value = weighted_loss(y_true, y_pred)
print(f"Custom class-based loss: {loss_value.numpy():.4f}")

# Using the lambda-based loss
lambda_loss = weighted_mse_mae_lambda(mse_weight=0.7, mae_weight=0.3)
loss_value = lambda_loss(y_true, y_pred)
print(f"Custom lambda-based loss: {loss_value.numpy():.4f}")

In [ ]:
# Custom loss function in PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Example 1: Class-based custom loss in PyTorch
class WeightedMSEMAELoss(nn.Module):
    def __init__(self, mse_weight=0.5, mae_weight=0.5):
        super(WeightedMSEMAELoss, self).__init__()
        self.mse_weight = mse_weight
        self.mae_weight = mae_weight
        
    def forward(self, y_pred, y_true):
        mse = torch.mean((y_pred - y_true) ** 2)
        mae = torch.mean(torch.abs(y_pred - y_true))
        return self.mse_weight * mse + self.mae_weight * mae

# Example 2: Functional custom loss in PyTorch
def weighted_mse_mae_func(y_pred, y_true, mse_weight=0.5, mae_weight=0.5):
    mse = torch.mean((y_pred - y_true) ** 2)
    mae = torch.mean(torch.abs(y_pred - y_true))
    return mse_weight * mse + mae_weight * mae

# Test the PyTorch losses
y_true = torch.tensor([1.0, 2.0, 3.0])
y_pred = torch.tensor([1.2, 1.8, 2.9])

# Using the class-based loss
criterion = WeightedMSEMAELoss(mse_weight=0.7, mae_weight=0.3)
loss_value = criterion(y_pred, y_true)
print(f"PyTorch class-based loss: {loss_value.item():.4f}")

# Using the functional loss
loss_value = weighted_mse_mae_func(y_pred, y_true, mse_weight=0.7, mae_weight=0.3)
print(f"PyTorch functional loss: {loss_value.item():.4f}")

### Custom Loss Example: Quantile Loss

Quantile loss is used in quantile regression to predict a specific quantile of the target distribution rather than the mean.

**Mathematical Formula**:
$$L_\tau(y, \hat{y}) = \sum_{i=1}^{n} \rho_\tau(y_i - \hat{y}_i)$$

Where:
$$\rho_\tau(u) = \begin{cases}
\tau \cdot u, & \text{if } u \geq 0 \\
(1 - \tau) \cdot (-u), & \text{if } u < 0
\end{cases}$$

$\tau$ is the desired quantile (e.g., 0.5 for median).

Let's implement this as a custom loss function:

In [ ]:
# Implementing Quantile Loss in TensorFlow
class QuantileLoss(tf.keras.losses.Loss):
    def __init__(self, quantile=0.5, name="quantile_loss"):
        super().__init__(name=name)
        self.quantile = quantile
        
    def call(self, y_true, y_pred):
        error = y_true - y_pred
        return tf.reduce_mean(tf.maximum(self.quantile * error, (self.quantile - 1) * error))

# Implementing Quantile Loss in PyTorch
class QuantileLossPyTorch(nn.Module):
    def __init__(self, quantile=0.5):
        super(QuantileLossPyTorch, self).__init__()
        self.quantile = quantile
        
    def forward(self, y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.max(
            self.quantile * error, 
            (self.quantile - 1) * error
        ))

# Test with different quantiles
y_true_np = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred_np = np.array([1.2, 1.8, 3.0, 4.5, 4.0])

quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]

print("Quantile Loss in TensorFlow:")
y_true_tf = tf.constant(y_true_np, dtype=tf.float32)
y_pred_tf = tf.constant(y_pred_np, dtype=tf.float32)

for q in quantiles:
    q_loss = QuantileLoss(quantile=q)
    loss_value = q_loss(y_true_tf, y_pred_tf)
    print(f"  Quantile = {q}: Loss = {loss_value.numpy():.4f}")

print("\nQuantile Loss in PyTorch:")
y_true_torch = torch.tensor(y_true_np, dtype=torch.float32)
y_pred_torch = torch.tensor(y_pred_np, dtype=torch.float32)

for q in quantiles:
    q_loss = QuantileLossPyTorch(quantile=q)
    loss_value = q_loss(y_pred_torch, y_true_torch)
    print(f"  Quantile = {q}: Loss = {loss_value.item():.4f}")

# Visualize Quantile Loss for different quantiles
plt.figure(figsize=(10, 6))
error_range = np.linspace(-2, 2, 100)

for q in quantiles:
    loss = np.maximum(q * error_range, (q - 1) * error_range)
    plt.plot(error_range, loss, label=f'τ = {q}')

plt.grid(True)
plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='-', alpha=0.3)
plt.title('Quantile Loss for Different Values of τ', fontsize=15)
plt.xlabel('Error (y_true - y_pred)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Loss Functions in Practice

Let's demonstrate how different loss functions affect model training and performance on a real dataset. We'll train regression models with different loss functions and compare their results.

In [ ]:
# Generate synthetic dataset with outliers
np.random.seed(42)
X = np.random.uniform(-5, 5, size=1000).reshape(-1, 1)
y_true = 0.5 * X[:, 0] + 2 + np.random.normal(0, 0.5, size=1000)

# Add some outliers
outlier_idx = np.random.choice(range(len(X)), size=50, replace=False)
y_true[outlier_idx] += np.random.normal(0, 5, size=50)

# Split data into train and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_true, test_size=0.2, random_state=42)

# Define a simple model for each loss function
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(16, activation='relu', input_shape=(1,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1)
    ])
    return model

# Train with different loss functions
losses = {
    'MSE': tf.keras.losses.MeanSquaredError(),
    'MAE': tf.keras.losses.MeanAbsoluteError(),
    'Huber': tf.keras.losses.Huber(delta=1.0),
    'LogCosh': tf.keras.losses.LogCosh()
}

# Dict to store models and results
trained_models = {}
histories = {}

# Train models with different losses
for loss_name, loss_func in losses.items():
    print(f"\nTraining model with {loss_name} loss...")
    model = create_model()
    model.compile(optimizer='adam', loss=loss_func)
    
    # Use early stopping to prevent overfitting
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=30, restore_best_weights=True)
    
    # Train the model
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        verbose=0,
        callbacks=[early_stop]
    )
    
    # Store model and history
    trained_models[loss_name] = model
    histories[loss_name] = history.history
    
    # Evaluate on test set
    test_loss = model.evaluate(X_test, y_test, verbose=0)
    print(f"  Test loss: {test_loss:.4f}")
    
    # Calculate additional metrics
    y_pred = model.predict(X_test, verbose=0).flatten()
    mse = mean_squared_error(y_test, y_pred)
    mae = np.mean(np.abs(y_test - y_pred))
    print(f"  Test MSE: {mse:.4f}")
    print(f"  Test MAE: {mae:.4f}")

# Plot training history
plt.figure(figsize=(12, 6))
for loss_name, history in histories.items():
    plt.plot(history['loss'], label=f'{loss_name} - Training')
    plt.plot(history['val_loss'], label=f'{loss_name} - Validation', linestyle='--')
plt.title('Training and Validation Loss Over Time', fontsize=15)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss Value', fontsize=12)
plt.legend()
plt.grid(True)
plt.show()

# Plot predictions vs actual values
plt.figure(figsize=(15, 10))
# Sort X for better line plots
sort_idx = np.argsort(X_test.flatten())
X_test_sorted = X_test[sort_idx]
y_test_sorted = y_test[sort_idx]

# Scatter plot of training data
plt.scatter(X_train, y_train, alpha=0.3, label='Training Data', color='gray')

# Plot each model's predictions
for i, (loss_name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_sorted, verbose=0).flatten()
    plt.plot(X_test_sorted, y_pred, linewidth=2, label=f'{loss_name} Predictions')

plt.title('Model Predictions with Different Loss Functions', fontsize=15)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Loss Function Selection Guide

Choosing the right loss function is crucial for the success of your machine learning model. Here's a guide to help you select the most appropriate loss function based on your problem:

### For Regression Tasks:
- **Mean Squared Error (MSE)**: Use when outliers are rare and errors follow a normal distribution.
- **Mean Absolute Error (MAE)**: Use when there may be outliers or when you want your model to be more robust.
- **Huber Loss**: A good compromise between MSE and MAE; use when you're unsure about the presence of outliers.
- **Log-cosh Loss**: Similar to Huber but with smoother derivatives; good for applications requiring second derivatives.
- **Quantile Loss**: When you need to predict a specific quantile of the target distribution (e.g., median).

### For Classification Tasks:
- **Binary Cross-Entropy**: For binary classification problems.
- **Categorical Cross-Entropy**: For multi-class classification with one-hot encoded labels.
- **Sparse Categorical Cross-Entropy**: For multi-class classification with integer labels.
- **Hinge Loss**: Good for maximum-margin classification algorithms like SVMs.
- **Focal Loss**: When dealing with severe class imbalance.

### For Special Use Cases:
- **IoU Loss**: For object detection and segmentation tasks.
- **Perceptual Loss**: For image generation tasks like super-resolution.
- **Adversarial Loss**: For generative adversarial networks.
- **Contrastive/Triplet Loss**: For similarity learning and embedding spaces.
- **CTC Loss**: For sequence-to-sequence problems without aligned inputs and outputs.

### General Selection Strategy:
1. Define your problem type (regression, classification, etc.)
2. Consider the data distribution and potential issues (outliers, imbalance)
3. Think about what "error" means in your specific domain
4. Choose a standard loss function as a starting point
5. Consider combining or customizing loss functions for your specific needs

## 9. Visualizing Loss Functions

Let's create visualizations of different loss functions to better understand their shapes, gradients, and behavior with respect to prediction errors. Visualizing loss functions can provide insights into how they penalize different types of errors.

In [ ]:
# Comparative visualization of multiple loss functions
def plot_loss_landscapes():
    # Create data for visualization
    prediction_range = np.linspace(-3, 3, 100)
    true_value = 1.0
    
    # Define loss functions
    losses = {
        'MSE': lambda y_true, y_pred: np.mean((y_true - y_pred)**2),
        'MAE': lambda y_true, y_pred: np.mean(np.abs(y_true - y_pred)),
        'Huber (δ=1.0)': lambda y_true, y_pred: huber_numpy(y_true, y_pred, delta=1.0),
        'Log-cosh': lambda y_true, y_pred: np.mean(np.log(np.cosh(y_true - y_pred)))
    }
    
    # Calculate loss values across prediction range
    loss_values = {}
    for name, loss_func in losses.items():
        loss_values[name] = [loss_func(true_value, pred) for pred in prediction_range]
    
    # Create plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    # 3D surface plot of regression losses
    X, Y = np.meshgrid(np.linspace(-3, 3, 50), np.linspace(-3, 3, 50))
    Z_mse = np.zeros_like(X)
    Z_mae = np.zeros_like(X)
    Z_huber = np.zeros_like(X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            y_true = X[i,j]
            y_pred = Y[i,j]
            Z_mse[i,j] = (y_true - y_pred)**2
            Z_mae[i,j] = np.abs(y_true - y_pred)
            Z_huber[i,j] = huber_numpy(np.array([y_true]), np.array([y_pred]), delta=1.0)
    
    ax = axes[0]
    surf = ax.plot_surface(X, Y, Z_mse, cmap='viridis', alpha=0.8)
    ax.set_title('MSE Loss Surface', fontsize=12)
    ax.set_xlabel('True Value')
    ax.set_ylabel('Predicted Value')
    ax.set_zlabel('Loss')
    
    # Plot loss functions for a fixed true value
    ax = axes[1]
    for name, values in loss_values.items():
        ax.plot(prediction_range, values, label=name, linewidth=2)
    
    ax.axvline(x=true_value, color='r', linestyle='--', label=f'True Value = {true_value}')
    ax.set_title('Regression Loss Functions', fontsize=12)
    ax.set_xlabel('Predicted Value')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True)
    
    # Plot classification losses
    ax = axes[2]
    probs = np.linspace(0.001, 0.999, 100)
    bce_true_1 = -np.log(probs)
    bce_true_0 = -np.log(1 - probs)
    
    ax.plot(probs, bce_true_1, label='BCE (true=1)', linewidth=2)
    ax.plot(probs, bce_true_0, label='BCE (true=0)', linewidth=2)
    
    # Add focal loss for comparison
    gamma = 2.0
    focal_true_1 = (1 - probs)**gamma * (-np.log(probs))
    focal_true_0 = probs**gamma * (-np.log(1 - probs))
    
    ax.plot(probs, focal_true_1, '--', label='Focal (γ=2, true=1)', linewidth=2)
    ax.plot(probs, focal_true_0, '--', label='Focal (γ=2, true=0)', linewidth=2)
    
    ax.set_title('Classification Loss Functions', fontsize=12)
    ax.set_xlabel('Predicted Probability for Class 1')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True)
    
    # Plot gradient of loss functions
    ax = axes[3]
    
    # Calculate gradients (derivatives) of loss functions
    mse_grad = 2 * (prediction_range - true_value)
    mae_grad = np.sign(prediction_range - true_value)
    
    # Huber gradient
    delta = 1.0
    error = prediction_range - true_value
    huber_grad = np.where(np.abs(error) <= delta, error, delta * np.sign(error))
    
    # Log-cosh gradient
    logcosh_grad = np.tanh(error)
    
    ax.plot(prediction_range, mse_grad, label='MSE gradient', linewidth=2)
    ax.plot(prediction_range, mae_grad, label='MAE gradient', linewidth=2)
    ax.plot(prediction_range, huber_grad, label='Huber gradient', linewidth=2)
    ax.plot(prediction_range, logcosh_grad, label='Log-cosh gradient', linewidth=2)
    
    ax.axvline(x=true_value, color='r', linestyle='--', label=f'True Value = {true_value}')
    ax.set_title('Gradients of Loss Functions', fontsize=12)
    ax.set_xlabel('Predicted Value')
    ax.set_ylabel('Gradient')
    ax.legend()
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

# Execute the visualization function
plot_loss_landscapes()

## Conclusion and Key Takeaways

Loss functions are fundamental components in deep learning that guide the optimization process and determine how models learn from their errors. In this notebook, we've explored a wide range of loss functions, their mathematical foundations, properties, and applications.

### Key Takeaways:

1. **Purpose of Loss Functions**:
   - Measure the difference between predictions and ground truth
   - Provide a differentiable objective for optimization
   - Guide model updates through backpropagation
   - Enable learning for specific tasks and data distributions

2. **Categories of Loss Functions**:
   - **Regression**: MSE, MAE, Huber Loss, Log-cosh Loss
   - **Classification**: Binary Cross-Entropy, Categorical Cross-Entropy, Hinge Loss, Focal Loss
   - **Special-purpose**: IoU Loss, Perceptual Loss, Contrastive/Triplet Loss

3. **Selection Criteria**:
   - Problem type (regression vs classification)
   - Data characteristics (presence of outliers, class imbalance)
   - Desired model behavior and robustness
   - Optimization considerations (smoothness, differentiability)

4. **Custom Loss Functions**:
   - Modern frameworks make it easy to implement custom losses
   - Combining losses can address multiple optimization objectives
   - Task-specific losses often outperform generic options

5. **Practical Considerations**:
   - Different loss functions can lead to significantly different model behaviors
   - The choice of loss function can impact convergence speed and stability
   - Some loss functions are more sensitive to hyperparameter selection
   - Understanding gradients is crucial for effective optimization

Loss functions are not merely implementation details but fundamental design choices that significantly impact model performance. By understanding the properties of different loss functions and their appropriate applications, you can make informed decisions in designing and training deep learning models for various tasks.

As deep learning research advances, new loss functions continue to be developed for specific problems and data types, highlighting the importance of this critical component in the machine learning pipeline.